In [27]:

from sqlalchemy import create_engine
import os
from dotenv import load_dotenv
load_dotenv('./../../.env')
load_dotenv('./../.env.secrets')
DB_TYPE = os.getenv('DB_TYPE', 'postgresql')
DB_PILOT = os.getenv('DB_PILOT', 'psycopg2')
DB_USER = 'kube'
DB_PASSWORD = os.getenv('DB_PASSWORD', 'password')
DB_HOST = os.getenv('DB_HOST', 'localhost')
DB_PORT = '5433'
DB_NAME = os.getenv('DB_NAME', 'optimsportbets-db')
DB_URL = f'{DB_TYPE}+{DB_PILOT}://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}'
engine = create_engine(DB_URL)

In [ ]:

import pandas as pd
from sqlalchemy import text

datetime_first_match = pd.to_datetime("2025-09-17 00:00:00")
model = "RSF_PR_LR"
sports_filter=[
                'soccer_fifa_club_world_cup', 'soccer_france_ligue_one', 'soccer_spain_la_liga',
                'soccer_italy_serie_a', 'soccer_germany_bundesliga', 'soccer_epl'
            ]

with engine.connect() as conn:
    # Rechargement des vues matérialisées
    # conn.execute(text("REFRESH MATERIALIZED VIEW mv_models_results_last_inferred;"))
    # conn.execute(text("REFRESH MATERIALIZED VIEW mv_soccer_odds_normalized;"))
    # conn.execute(text("REFRESH MATERIALIZED VIEW mv_soccer_odds_last_normalized;"))

    df_models_results = pd.read_sql(
            text("""
                SELECT *
                FROM mv_models_results_last_inferred
                WHERE model = :model
                  AND date_match >= :date_match
            """),
            conn,
            params={
                "model": model,
                "date_match": datetime_first_match.date()
            }
        )

    df_odds = pd.read_sql(
            text("""
                SELECT *
                FROM mv_soccer_odds_last_normalized
                WHERE commence_time >= :datetime_first_match
            """),
            conn,
            params={"datetime_first_match": datetime_first_match}
        )

    # if bookmakers:

    # if sports_filter:
    #     df_odds = df_odds[df_odds['sport_key'].isin(sports_filter)]

    # Pivot home/draw/away
    df_home = df_odds[df_odds['outcome_name'] == df_odds['home_team']].copy()
    df_draw = df_odds[df_odds['outcome_name'] == 'Draw'].copy()
    df_away = df_odds[df_odds['outcome_name'] == df_odds['away_team']].copy()

    df_home = df_home.rename(columns={'outcome_price': 'odds_home', 'bookmaker_title': 'bookmaker_home', 'bookmaker_last_update': 'odds_home_datetime'})
    df_draw = df_draw.rename(columns={'outcome_price': 'odds_draw', 'bookmaker_title': 'bookmaker_draw', 'bookmaker_last_update': 'odds_draw_datetime'})
    df_away = df_away.rename(columns={'outcome_price': 'odds_away', 'bookmaker_title': 'bookmaker_away', 'bookmaker_last_update': 'odds_away_datetime'})
    df_home['bookmaker_home_key'] = df_home['bookmaker_key']
    df_draw['bookmaker_draw_key'] = df_draw['bookmaker_key']
    df_away['bookmaker_away_key'] = df_away['bookmaker_key']

    df_home = df_home[['match_id', 'bookmaker_key', 'odds_home', 'odds_home_datetime', 'bookmaker_home', 'bookmaker_home_key']]
    df_draw = df_draw[['match_id', 'bookmaker_key', 'odds_draw', 'odds_draw_datetime', 'bookmaker_draw', 'bookmaker_draw_key']]
    df_away = df_away[['match_id', 'bookmaker_key', 'odds_away', 'odds_away_datetime', 'bookmaker_away', 'bookmaker_away_key']]

    df_pivot = df_home.merge(df_draw, on=['match_id', 'bookmaker_key']).merge(df_away, on=['match_id', 'bookmaker_key'])

    def get_best(df, col):
        return df.loc[df[col].idxmax()]

    best_odds = df_pivot.groupby('match_id').apply(lambda g: pd.Series({
        'odds_home': g['odds_home'].max(),
        'odds_draw': g['odds_draw'].max(),
        'odds_away': g['odds_away'].max(),
        'bookmaker_home': get_best(g, 'odds_home')['bookmaker_home'],
        'bookmaker_draw': get_best(g, 'odds_draw')['bookmaker_draw'],
        'bookmaker_away': get_best(g, 'odds_away')['bookmaker_away'],
        'bookmaker_home_key': get_best(g, 'odds_home')['bookmaker_home_key'],
        'bookmaker_draw_key': get_best(g, 'odds_draw')['bookmaker_draw_key'],
        'bookmaker_away_key': get_best(g, 'odds_away')['bookmaker_away_key'],
        'odds_home_datetime': get_best(g, 'odds_home')['odds_home_datetime'],
        'odds_draw_datetime': get_best(g, 'odds_draw')['odds_draw_datetime'],
        'odds_away_datetime': get_best(g, 'odds_away')['odds_away_datetime'],
    })).reset_index()

    df_match_info = df_odds.drop_duplicates(subset='match_id')[
        ['match_id', 'home_team_canonical', 'away_team_canonical', 'commence_time', 'sport_key']
    ]
    df_odds_final = best_odds.merge(df_match_info, on='match_id', how='left')
    df_odds_final['date_match'] = df_odds_final['commence_time'].dt.date

    df_final = df_models_results.merge(
        df_odds_final,
        left_on=['home_team', 'away_team', 'date_match'],
        right_on=['home_team_canonical', 'away_team_canonical', 'date_match'],
        how='inner'
    )

    df_final

/var/folders/3f/cb7qvm7j7y92h0xjlhhz115h0000gn/T/ipykernel_20214/3251272343.py:68: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  best_odds = df_pivot.groupby('match_id').apply(lambda g: pd.Series({


In [39]:
df_models_results

,datetime_inference,model,game_id,game,date_match,time_match,home_team,away_team,prob_home_win,prob_draw,prob_away_win,prob_home_team_score,prob_away_team_score,rk
0,2025-09-16 15:47:02.571962,RSF_PR_LR,None,2025-09-17 Bayern Munich-Chelsea,2025-09-17,21:00:00,Bayern Munich,Chelsea,0.518562,0.198526,0.282912,None,None,1
1,2025-09-16 15:47:02.571962,RSF_PR_LR,None,2025-09-17 Liverpool-Atlético Madrid,2025-09-17,20:00:00,Liverpool,Atlético Madrid,0.474505,0.242081,0.283414,None,None,1
2,2025-09-16 15:47:02.571962,RSF_PR_LR,None,2025-09-17 Paris S-G-Atalanta,2025-09-17,21:00:00,Paris S-G,Atalanta,0.543724,0.224358,0.231918,None,None,1
3,2025-09-16 15:47:02.571962,RSF_PR_LR,None,2025-09-18 Manchester City-Napoli,2025-09-18,20:00:00,Manchester City,Napoli,0.510924,0.216910,0.272166,None,None,1
4,2025-09-16 15:47:02.571962,RSF_PR_LR,None,2025-09-18 Newcastle Utd-Barcelona,2025-09-18,20:00:00,Newcastle Utd,Barcelona,0.182130,0.215972,0.601898,None,None,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1648,2025-09-16 15:47:02.571962,RSF_PR_LR,None,2026-05-24 Torino-Juventus,2026-05-24,None,Torino,Juventus,0.154367,0.224767,0.620866,None,None,1
1649,2025-09-16 15:47:02.571962,RSF_PR_LR,None,2026-05-24 Tottenham-Everton,2026-05-24,16:00:00,Tottenham,Everton,0.512148,0.260334,0.227518,None,None,1
1650,2025-09-16 15:47:02.571962,RSF_PR_LR,None,2026-05-24 Valencia-Barcelona,2026-05-24,None,Valencia,Barcelona,0.105569,0.185883,0.708548,None,None,1
1651,2025-09-16 15:47:02.571962,RSF_PR_LR,None,2026-05-24 Villarreal-Atlético Madrid,2026-05-24,None,Villarreal,Atlético Madrid,0.240439,0.239271,0.520290,None,None,1


In [35]:
df_odds_final.sort_values(by='commence_time').head(10)

,match_id,odds_home,odds_draw,odds_away,bookmaker_home,bookmaker_draw,bookmaker_away,bookmaker_home_key,bookmaker_draw_key,bookmaker_away_key,odds_home_datetime,odds_draw_datetime,odds_away_datetime,home_team_canonical,away_team_canonical,commence_time,sport_key,date_match
48,66293041d878fd7670b058dc25d0c089,1.45,5.20,10.00,1xBet,Betfair,Betfair,onexbet,betfair_ex_eu,betfair_ex_uk,2025-09-16 15:16:48,2025-09-16 15:16:46,2025-09-16 15:16:46,Olympiacos,Pafos FC,2025-09-17 16:45:00,soccer_uefa_champs_league,2025-09-17
92,d091291c7bd11dac67b0c14b6ce3b458,1.85,4.50,4.40,1xBet,Betfair,Betfair,onexbet,betfair_ex_eu,betfair_ex_uk,2025-09-16 15:16:48,2025-09-16 15:16:46,2025-09-16 15:16:46,Slavia Prague,Bodø/Glimt,2025-09-17 16:45:00,soccer_uefa_champs_league,2025-09-17
66,8d3aa404e7c308b261365df88e4ef927,4.40,4.10,1.94,Betfair,Betfair,Betfair,betfair_ex_eu,betfair_ex_eu,betfair_ex_eu,2025-09-16 15:16:46,2025-09-16 15:16:46,2025-09-16 15:16:46,Ajax,Inter Milan,2025-09-17 19:00:00,soccer_uefa_champs_league,2025-09-17
98,d6b2f14af3e5b680e48eaf76bfd8287d,1.51,5.20,8.00,1xBet,Betfair,Matchbook,onexbet,betfair_ex_uk,matchbook,2025-09-16 15:16:48,2025-09-16 15:16:46,2025-09-16 15:16:46,Paris Saint-Germain,Atalanta,2025-09-17 19:00:00,soccer_uefa_champs_league,2025-09-17
85,c0f5bd160c6b24e71e661f1f2f948512,1.71,4.70,5.20,Betfair,Betfair,Betfair,betfair_ex_eu,betfair_ex_eu,betfair_ex_eu,2025-09-16 15:16:46,2025-09-16 15:16:46,2025-09-16 15:16:46,Bayern Munich,Chelsea,2025-09-17 19:00:00,soccer_uefa_champs_league,2025-09-17
114,f0fb8f95e2ca15f59ee7f7b72624fb74,1.58,4.80,7.40,BetAnySports,Matchbook,Betfair,betanysports,matchbook,betfair_ex_eu,2025-09-16 09:49:44,2025-09-16 15:16:46,2025-09-16 15:16:46,Liverpool,Atlético Madrid,2025-09-17 19:00:00,soccer_uefa_champs_league,2025-09-17
25,395a8859dd2297e5195171cf80a4e537,3.60,3.75,2.30,Betfair,1xBet,Betfair,betfair_ex_eu,onexbet,betfair_ex_eu,2025-09-16 15:16:46,2025-09-16 15:16:48,2025-09-16 15:16:46,FC Copenhagen,Bayer Leverkusen,2025-09-18 16:45:00,soccer_uefa_champs_league,2025-09-18
94,d4488ec9d080a2fff4ca51bfe25e50ee,2.88,3.85,2.62,Smarkets,Betfair,Betfair,smarkets,betfair_ex_eu,betfair_ex_eu,2025-09-16 15:16:31,2025-09-16 15:16:46,2025-09-16 15:16:46,Club Brugge,Monaco,2025-09-18 16:45:00,soccer_uefa_champs_league,2025-09-18
1,0f0b6988e5365d7c31a5d4c6ed9d182a,1.72,4.40,5.60,Betfair,Unibet (NL),Betfair,betfair_ex_eu,unibet_nl,betfair_ex_eu,2025-09-16 15:16:46,2025-09-16 15:16:49,2025-09-16 15:16:46,Manchester City,Napoli,2025-09-18 19:00:00,soccer_uefa_champs_league,2025-09-18
11,1f6548dcf8b047a1feee6be76685b303,3.04,4.00,2.48,1xBet,Matchbook,Betfair,onexbet,matchbook,betfair_ex_eu,2025-09-16 15:16:48,2025-09-16 15:16:46,2025-09-16 15:16:46,Newcastle United,Barcelona,2025-09-18 19:00:00,soccer_uefa_champs_league,2025-09-18


In [29]:
df_final

,datetime_inference,model,game_id,game,date_match,time_match,home_team,away_team,prob_home_win,prob_draw,...,bookmaker_home_key,bookmaker_draw_key,bookmaker_away_key,odds_home_datetime,odds_draw_datetime,odds_away_datetime,home_team_canonical,away_team_canonical,commence_time,sport_key
0,2025-09-16 15:47:02.571962,RSF_PR_LR,None,2025-09-16 Athletic Club-Arsenal,2025-09-16,18:45:00,Athletic Club,Arsenal,0.201268,0.225847,...,betfair_ex_eu,suprabets,onexbet,2025-09-16 15:16:46,2025-09-16 15:11:54,2025-09-16 15:16:48,Athletic Club,Arsenal,2025-09-16 16:45:00,soccer_uefa_champs_league
1,2025-09-16 15:47:02.571962,RSF_PR_LR,None,2025-09-16 Real Madrid-Marseille,2025-09-16,21:00:00,Real Madrid,Marseille,0.634424,0.198826,...,sport888,betfair_ex_eu,betfair_ex_eu,2025-09-16 15:16:48,2025-09-16 15:16:46,2025-09-16 15:16:46,Real Madrid,Marseille,2025-09-16 19:00:00,soccer_uefa_champs_league
2,2025-09-16 15:47:02.571962,RSF_PR_LR,None,2025-09-17 Bayern Munich-Chelsea,2025-09-17,21:00:00,Bayern Munich,Chelsea,0.518562,0.198526,...,betfair_ex_eu,betfair_ex_eu,betfair_ex_eu,2025-09-16 15:16:46,2025-09-16 15:16:46,2025-09-16 15:16:46,Bayern Munich,Chelsea,2025-09-17 19:00:00,soccer_uefa_champs_league
3,2025-09-16 15:47:02.571962,RSF_PR_LR,None,2025-09-17 Liverpool-Atlético Madrid,2025-09-17,20:00:00,Liverpool,Atlético Madrid,0.474505,0.242081,...,betanysports,matchbook,betfair_ex_eu,2025-09-16 09:49:44,2025-09-16 15:16:46,2025-09-16 15:16:46,Liverpool,Atlético Madrid,2025-09-17 19:00:00,soccer_uefa_champs_league
4,2025-09-16 15:47:02.571962,RSF_PR_LR,None,2025-09-18 Manchester City-Napoli,2025-09-18,20:00:00,Manchester City,Napoli,0.510924,0.216910,...,betfair_ex_eu,unibet_nl,betfair_ex_eu,2025-09-16 15:16:46,2025-09-16 15:16:49,2025-09-16 15:16:46,Manchester City,Napoli,2025-09-18 19:00:00,soccer_uefa_champs_league
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
76,2025-09-16 15:47:02.571962,RSF_PR_LR,None,2025-09-28 Roma-Hellas Verona,2025-09-28,15:00:00,Roma,Hellas Verona,0.577725,0.255595,...,onexbet,onexbet,smarkets,2025-09-16 15:16:50,2025-09-16 15:16:50,2025-09-16 15:16:51,Roma,Hellas Verona,2025-09-28 13:00:00,soccer_italy_serie_a
77,2025-09-16 15:47:02.571962,RSF_PR_LR,None,2025-09-28 Sassuolo-Udinese,2025-09-28,12:30:00,Sassuolo,Udinese,0.373903,0.300155,...,onexbet,unibet_nl,tipico_de,2025-09-16 15:16:50,2025-09-16 15:15:53,2025-09-16 15:16:03,Sassuolo,Udinese,2025-09-28 10:30:00,soccer_italy_serie_a
78,2025-09-16 15:47:02.571962,RSF_PR_LR,None,2025-09-28 Union Berlin-Hamburger SV,2025-09-28,19:30:00,Union Berlin,Hamburger SV,0.407223,0.301862,...,betfair_ex_eu,smarkets,betfair_ex_eu,2025-09-16 15:16:42,2025-09-16 15:16:43,2025-09-16 15:16:42,Union Berlin,Hamburger SV,2025-09-28 17:30:00,soccer_germany_bundesliga
79,2025-09-16 15:47:02.571962,RSF_PR_LR,None,2025-09-29 Genoa-Lazio,2025-09-29,20:45:00,Genoa,Lazio,0.146571,0.249066,...,onexbet,smarkets,smarkets,2025-09-16 15:16:50,2025-09-16 15:16:51,2025-09-16 15:16:51,Genoa,Lazio,2025-09-29 18:45:00,soccer_italy_serie_a


In [24]:
same_day = True
with engine.connect() as conn:

    conn.execute(text("REFRESH MATERIALIZED VIEW mv_models_results_last_inferred;"))
    conn.execute(text("REFRESH MATERIALIZED VIEW mv_soccer_odds_normalized;"))
    conn.execute(text("REFRESH MATERIALIZED VIEW mv_soccer_odds_last_normalized;"))
    conn.execute(text("REFRESH MATERIALIZED VIEW mv_models_results_last_inferred;"))

    if same_day:
        datetime_first_match = datetime_first_match.replace(hour=0, minute=0, second=0, microsecond=0)
        datetime_last_match = datetime_first_match + pd.Timedelta(days=1)
        
        # Query pour les résultats du modèle - jour même uniquement
        df_models_results = pd.read_sql(
            text("""
                SELECT *
                FROM mv_models_results_last_inferred
                WHERE model = :model
                    AND date_match = :date_match
            """),
            conn,
            params={
                "model": model,
                "date_match": datetime_first_match.date()
            }
        )

        # Query pour les cotes - jour même uniquement
        df_odds = pd.read_sql(
            text("""
                SELECT *
                FROM mv_soccer_odds_last_normalized
                WHERE commence_time >= :datetime_first_match
                    AND commence_time < :datetime_last_match
            """),
            conn,
            params={
                "datetime_first_match": datetime_first_match,
                "datetime_last_match": datetime_last_match
            }
        )
    else:
        datetime_last_match = None
        
        # Query pour les résultats du modèle - à partir de la date donnée
        df_models_results = pd.read_sql(
            text("""
                SELECT *
                FROM mv_models_results_last_inferred
                WHERE model = :model
                    AND date_match >= :date_match
            """),
            conn,
            params={
                "model": model,
                "date_match": datetime_first_match.date()
            }
        )

        # Query pour les cotes - à partir de la datetime donnée
        df_odds = pd.read_sql(
            text("""
                SELECT *
                FROM mv_soccer_odds_last_normalized
                WHERE commence_time >= :datetime_first_match
            """),
            conn,
            params={"datetime_first_match": datetime_first_match}
        )

# if bookmakers:
#     df_odds = df_odds[df_odds['bookmaker_key'].isin(bookmakers)]

# if sports_filter:
#     df_odds = df_odds[df_odds['sport_key'].isin(sports_filter)]

# Pivot home/draw/away
df_home = df_odds[df_odds['outcome_name'] == df_odds['home_team_canonical']].copy()
df_draw = df_odds[df_odds['outcome_name'] == 'Draw'].copy()
df_away = df_odds[df_odds['outcome_name'] == df_odds['away_team_canonical']].copy()

df_home = df_home.rename(columns={'outcome_price': 'odds_home', 'bookmaker_title': 'bookmaker_home', 'bookmaker_last_update': 'odds_home_datetime'})
df_draw = df_draw.rename(columns={'outcome_price': 'odds_draw', 'bookmaker_title': 'bookmaker_draw', 'bookmaker_last_update': 'odds_draw_datetime'})
df_away = df_away.rename(columns={'outcome_price': 'odds_away', 'bookmaker_title': 'bookmaker_away', 'bookmaker_last_update': 'odds_away_datetime'})
df_home['bookmaker_home_key'] = df_home['bookmaker_key']
df_draw['bookmaker_draw_key'] = df_draw['bookmaker_key']
df_away['bookmaker_away_key'] = df_away['bookmaker_key']

df_home = df_home[['match_id', 'bookmaker_key', 'odds_home', 'odds_home_datetime', 'bookmaker_home', 'bookmaker_home_key']]
df_draw = df_draw[['match_id', 'bookmaker_key', 'odds_draw', 'odds_draw_datetime', 'bookmaker_draw', 'bookmaker_draw_key']]
df_away = df_away[['match_id', 'bookmaker_key', 'odds_away', 'odds_away_datetime', 'bookmaker_away', 'bookmaker_away_key']]

df_pivot = df_home.merge(df_draw, on=['match_id', 'bookmaker_key']).merge(df_away, on=['match_id', 'bookmaker_key'])

def get_best(df, col):
    return df.loc[df[col].idxmax()]

best_odds = df_pivot.groupby('match_id').apply(lambda g: pd.Series({
    'odds_home': g['odds_home'].max(),
    'odds_draw': g['odds_draw'].max(),
    'odds_away': g['odds_away'].max(),
    'bookmaker_home': get_best(g, 'odds_home')['bookmaker_home'],
    'bookmaker_draw': get_best(g, 'odds_draw')['bookmaker_draw'],
    'bookmaker_away': get_best(g, 'odds_away')['bookmaker_away'],
    'bookmaker_home_key': get_best(g, 'odds_home')['bookmaker_home_key'],
    'bookmaker_draw_key': get_best(g, 'odds_draw')['bookmaker_draw_key'],
    'bookmaker_away_key': get_best(g, 'odds_away')['bookmaker_away_key'],
    'odds_home_datetime': get_best(g, 'odds_home')['odds_home_datetime'],
    'odds_draw_datetime': get_best(g, 'odds_draw')['odds_draw_datetime'],
    'odds_away_datetime': get_best(g, 'odds_away')['odds_away_datetime'],
})).reset_index()


df_match_info = df_odds.drop_duplicates(subset='match_id')[
    ['match_id', 'home_team_canonical', 'away_team_canonical', 'commence_time', 'sport_key']
]
df_odds_final = best_odds.merge(df_match_info, on='match_id', how='left')
df_odds_final['date_match'] = df_odds_final['commence_time'].dt.date

df_final = df_models_results.merge(
    df_odds_final,
    left_on=['home_team', 'away_team', 'date_match'],
    right_on=['home_team_canonical', 'away_team_canonical', 'date_match'],
    how='inner'
)

/var/folders/3f/cb7qvm7j7y92h0xjlhhz115h0000gn/T/ipykernel_20214/3415125741.py:98: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  best_odds = df_pivot.groupby('match_id').apply(lambda g: pd.Series({


In [25]:
df_final

,datetime_inference,model,game_id,game,date_match,time_match,home_team,away_team,prob_home_win,prob_draw,...,bookmaker_home_key,bookmaker_draw_key,bookmaker_away_key,odds_home_datetime,odds_draw_datetime,odds_away_datetime,home_team_canonical,away_team_canonical,commence_time,sport_key
0,2025-09-16 15:47:02.571962,RSF_PR_LR,None,2025-09-16 Real Madrid-Marseille,2025-09-16,21:00:00,Real Madrid,Marseille,0.634424,0.198826,...,sport888,betfair_ex_eu,betfair_ex_eu,2025-09-16 15:16:48,2025-09-16 15:16:46,2025-09-16 15:16:46,Real Madrid,Marseille,2025-09-16 19:00:00,soccer_uefa_champs_league
